# a) Core GA Functions

In [15]:
import random

def decode(chromosome):
    return int("".join(map(str, chromosome)), 2)

def fitness(chromosome):
    x = decode(chromosome)
    return -x**2 + 14*x + 5

def roulette_select(population):
    fitness_values = [fitness(ind) for ind in population]
    total = sum(fitness_values)

    r = random.random()
    cumulative = 0

    for ind, fit in zip(population, fitness_values):
        cumulative += fit / total
        if r <= cumulative:
            return ind

def single_point_crossover(p1, p2, point):
    return (
        p1[:point] + p2[point:],
        p2[:point] + p1[point:]
    )

def mutate(chromosome, pm):
    return [
        1 - bit if random.random() < pm else bit
        for bit in chromosome
    ]

In [16]:
def test_functions():
    test_cases = [
        [0,1,1,0],
        [1,0,0,1],
        [1,1,0,0],
        [0,0,1,1]
    ]

    print("Testing Decode & Fitness:\n")
    for c in test_cases:
        x = decode(c)
        f = fitness(c)
        print(f"{c} -> x={x}, f(x)={f}")

test_functions()

Testing Decode & Fitness:

[0, 1, 1, 0] -> x=6, f(x)=53
[1, 0, 0, 1] -> x=9, f(x)=50
[1, 1, 0, 0] -> x=12, f(x)=29
[0, 0, 1, 1] -> x=3, f(x)=38


# b) Full GA

In [17]:
def random_chromosome():
    return [random.randint(0,1) for _ in range(4)]

def run_ga(pop_size, num_generations, pm, elitism=False, verbose=False):
    population = [random_chromosome() for _ in range(pop_size)]

    history = []

    for gen in range(num_generations):
        fitness_vals = [fitness(ind) for ind in population]

        best_idx = fitness_vals.index(max(fitness_vals))
        best = population[best_idx]
        best_fit = fitness_vals[best_idx]

        history.append((gen, best_fit, decode(best)))

        if verbose:
            print(f"\nGeneration {gen}")
            print("Population:", population)
            print("x values:", [decode(ind) for ind in population])
            print("Fitness:", fitness_vals)
            print("Best:", best, "x=", decode(best), "f=", best_fit)

        new_population = []

        if elitism:
            new_population.append(best)

        while len(new_population) < pop_size:
            p1 = roulette_select(population)
            p2 = roulette_select(population)

            point = random.randint(1, 3)
            c1, c2 = single_point_crossover(p1, p2, point)

            c1 = mutate(c1, pm)
            c2 = mutate(c2, pm)

            new_population.extend([c1, c2])

        population = new_population[:pop_size]

    return history

In [18]:
print("\n=== GA RUN (Required Output) ===")
run_ga(pop_size=4, num_generations=10, pm=0.1, elitism=False, verbose=True)


=== GA RUN (Required Output) ===

Generation 0
Population: [[0, 0, 1, 1], [1, 0, 1, 0], [1, 1, 0, 0], [0, 0, 0, 1]]
x values: [3, 10, 12, 1]
Fitness: [38, 45, 29, 18]
Best: [1, 0, 1, 0] x= 10 f= 45

Generation 1
Population: [[1, 0, 1, 0], [1, 0, 1, 0], [0, 0, 0, 1], [0, 0, 1, 1]]
x values: [10, 10, 1, 3]
Fitness: [45, 45, 18, 38]
Best: [1, 0, 1, 0] x= 10 f= 45

Generation 2
Population: [[1, 0, 1, 0], [1, 1, 1, 1], [1, 0, 0, 0], [0, 0, 1, 0]]
x values: [10, 15, 8, 2]
Fitness: [45, -10, 53, 29]
Best: [1, 0, 0, 0] x= 8 f= 53

Generation 3
Population: [[0, 0, 0, 0], [1, 0, 1, 0], [1, 0, 1, 0], [1, 0, 1, 1]]
x values: [0, 10, 10, 11]
Fitness: [5, 45, 45, 38]
Best: [1, 0, 1, 0] x= 10 f= 45

Generation 4
Population: [[1, 0, 1, 1], [1, 0, 1, 0], [1, 1, 0, 1], [1, 0, 1, 1]]
x values: [11, 10, 13, 11]
Fitness: [38, 45, 18, 38]
Best: [1, 0, 1, 0] x= 10 f= 45

Generation 5
Population: [[1, 0, 1, 0], [0, 1, 1, 1], [1, 0, 1, 0], [1, 1, 1, 1]]
x values: [10, 7, 10, 15]
Fitness: [45, 54, 45, -10]
Bes

[(0, 45, 10),
 (1, 45, 10),
 (2, 53, 8),
 (3, 45, 10),
 (4, 45, 10),
 (5, 54, 7),
 (6, 45, 10),
 (7, 50, 5),
 (8, 53, 6),
 (9, 53, 6)]

# c) Experiments

In [19]:
def experiment_elitism():
    trials = 30

    for elitism in [False, True]:
        best_fitness_total = 0
        success = 0
        gen_to_50_list = []

        for _ in range(trials):
            history = run_ga(4, 20, 0.1, elitism, verbose=False)

            best_fit = max([h[1] for h in history])
            best_fitness_total += best_fit

            reached = False
            for g, f, x in history:
                if f >= 50:
                    gen_to_50_list.append(g)
                    reached = True
                    break

            if any(x == 7 for _, _, x in history):
                success += 1

        avg_best = best_fitness_total / trials
        avg_gen = sum(gen_to_50_list)/len(gen_to_50_list) if gen_to_50_list else 0

        print(f"\nElitism={elitism}")
        print("Avg Best Fitness:", round(avg_best,2))
        print("Runs reaching x=7:", success)
        print("Avg Gen to f>=50:", round(avg_gen,2))

In [20]:
def experiment_mutation():
    trials = 30
    pms = [0.01, 0.1, 0.3, 0.5]

    print("\nMutation Experiment Results:\n")

    for pm in pms:
        total = 0

        for _ in range(trials):
            history = run_ga(4, 20, pm, verbose=False)
            best = max([h[1] for h in history])
            total += best

        print(f"pm={pm}, Avg Best Fitness={round(total/trials,2)}")

In [21]:
print("\n=== Elitism Experiment ===")
experiment_elitism()

print("\n=== Mutation Experiment ===")
experiment_mutation()


=== Elitism Experiment ===

Elitism=False
Avg Best Fitness: 53.97
Runs reaching x=7: 29
Avg Gen to f>=50: 0.67

Elitism=True
Avg Best Fitness: 53.7
Runs reaching x=7: 21
Avg Gen to f>=50: 1.23

=== Mutation Experiment ===

Mutation Experiment Results:

pm=0.01, Avg Best Fitness=52.1
pm=0.1, Avg Best Fitness=53.67
pm=0.3, Avg Best Fitness=53.97
pm=0.5, Avg Best Fitness=54.0
